<a href="https://colab.research.google.com/github/kannanrk28/Capstone_Masai_FinalProject_Kannan/blob/main/part1/Part1_DataPreparationProcess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part 1 — Data Acquisition, Cleaning, and Exploratory Analysis

In [ ]:
import pandas as pd

# Read Dataset from Github repo
URL = "https://raw.githubusercontent.com/kannanrk28/Capstone_Masai_FinalProject_Kannan/main/dataset/tmdb-movies.csv"
df_movie = pd.read_csv(URL)

#Printing laptop data shape and data types
print(df_movie.shape)
print(df_movie.dtypes)
print(df_movie.head(5))

In [ ]:
#Finding null count
null_count=df_movie.isnull().sum()
print(f'Null Count:{null_count}')

In [ ]:
#Finding null percentage
null_percentage=((df_movie.isnull().sum()/df_movie.shape[0])*100)
print(f'Null Percentage:{null_percentage}')

In [ ]:
#Finding columns which has null values greater than 20%
col_with_null_values=null_percentage[null_percentage>20]
if len(col_with_null_values) == 0:
    print('No columns have null values greater than 20%.')
else:
    print(f'Columns with null values greater than 20%:{col_with_null_values}')

In [ ]:
# Convert binary columns to boolean
df_clean=df_movie.copy()

# Find categorical columns (release_date will no longer be included)
categorical_columns = df_clean.select_dtypes(include=['object']).columns

print("Categorical Columns:")
print(categorical_columns.tolist())

# Find boolean columns
boolean_columns = df_clean.select_dtypes(include=['bool']).columns

print("\nBoolean Columns:")
print(boolean_columns.tolist())
print(df_clean.dtypes)

In [ ]:
# Select numeric columns

numeric_columns = df_clean.select_dtypes(include=['int64', 'float64']).columns
counter =1
# Impute missing values using median
for col in numeric_columns:
    null_percentage = (df_clean[col].isnull().sum() / len(df_clean)) * 100

    if 0 < null_percentage < 20:
        counter= counter +1
        median_value = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_value)

        print(f"Filled missing values in '{col}' using median: {median_value}")

if(counter == 1) :
  print(f"This data set does not contain any null values for the data type Integer or Float")

In [ ]:
#Finding duplicates and removing it
total_rows = df_clean.shape[0]
print(f"Total rows in the Movies DataFrame: {total_rows}")
duplicates_count=df_clean.duplicated().sum()
print(f"Total duplicate rows in the Movies DataFrame: {duplicates_count}")
df_clean=df_clean.drop_duplicates()
print(f"Total rows after dropping duplicates: {df_clean.shape[0]}")
print(f'Initial Null Percentage:{null_percentage}')
null_percentage_df_clean = (df_clean[col].isnull().sum() / df_clean.shape[0]) * 100
print(f'Null Percentage after dropping duplicates:{null_percentage_df_clean}')
df_clean.shape

In [ ]:
# 4. Data type correction:
# Convert release_date to datetime
df_clean['release_date'] = pd.to_datetime(df_clean['release_date'], errors='coerce')

# Memory usage before data type conversion
memory_before = df_clean.memory_usage(deep=True).sum()
print(f"Memory usage before conversion: {memory_before} bytes")

# Check original data types
print("\nData types before conversion:")
# print(df_clean.dtypes)

# Convert repetitive string columns to category
category_cols = ['genres', 'director']

for col in category_cols:
    df_clean[col] = df_clean[col].astype('category')

# Memory after conversion
memory_after = df_clean.memory_usage(deep=True).sum()

print(f"Memory after : {memory_after:,} bytes")
print(f"Memory saved : {memory_before - memory_after:,} bytes")
print(f"Justification:\nConverting genres and director columns to category type did not show any change in memory optimization, that's because this particular dataset doesn't have many low-cardinality object columns.")

In [ ]:
# 5.Descriptive statistics and skewness:

# Select numeric columns
numeric_columns = df_clean.select_dtypes(include=['int64', 'float64']).columns

# Descriptive statistics
print("Descriptive Statistics:")
print(df_clean[numeric_columns].describe())

# Calculate skewness for each numeric column
skewness = df_clean[numeric_columns].skew()

print("\nSkewness of Numeric Columns:")
print(skewness.sort_values(key=abs, ascending=False))

# Find the column with the highest absolute skewness
highest_skew_col = skewness.abs().idxmax()
highest_skew_value = skewness[highest_skew_col]

print(f"\nColumn with highest absolute skewness: {highest_skew_col}")
print(f"Skewness value: {highest_skew_value:.3f}")

In [ ]:
# 6. Outlier detection with IQR:

# Numeric columns to analyze
columns = ['popularity', 'revenue']

for col in columns:
    # Calculate Q1, Q3 and IQR
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1

    # Calculate lower and upper bounds
    lower_bound = Q1 - (1.5 * IQR)
    upper_bound = Q3 + (1.5 * IQR)

    # Count outliers
    outliers = df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)]

    print(f"\nColumn: {col}")
    print(f"Q1           : {Q1:.2f}")
    print(f"Q3           : {Q3:.2f}")
    print(f"IQR          : {IQR:.2f}")
    print(f"Lower Bound  : {lower_bound:.2f}")
    print(f"Upper Bound  : {upper_bound:.2f}")
    print(f"Outliers     : {len(outliers)}")

In [ ]:
# 7.Visualization - Line chart
import matplotlib.pyplot as plt

# Calculate average popularity by release year
popularity_by_year = (
    df_clean.groupby('release_year')['popularity']
    .mean()
    .reset_index()
)

# Create the line plot
plt.figure(figsize=(12, 6))

plt.plot(
    popularity_by_year['release_year'],
    popularity_by_year['popularity'],
    marker='o',
    linestyle='-',
    linewidth=2,
    markersize=8,
    color='blue'
)

# Add title and labels
plt.title('Movie Popularity Trend Over the Years', fontsize=18)
plt.xlabel('Release Year', fontsize=14)
plt.ylabel('Average Popularity', fontsize=14)

# Add grid
plt.grid(True, linestyle='--', alpha=0.6)

# Show the plot
plt.show()

In [ ]:
# 7.Visualization - Bar chart
import matplotlib.pyplot as plt

# # Extract the primary genre (first genre listed)
# df_clean['primary_genre'] = df_clean['genres'].str.split('|').str[0]

# Calculate the average popularity for each genre
genre_popularity = (
    df_clean.groupby('genres')['popularity']
    .mean()
    .sort_values(ascending=False)
    .head(10)   # Show only top 10 combinations
)

# Create bar chart
plt.figure(figsize=(12, 6))

plt.bar(
    genre_popularity.index,
    genre_popularity.values,
    color='skyblue',
    edgecolor='black'
)

# Add title and labels
plt.title('Average Movie Popularity by Genre', fontsize=16)
plt.xlabel('Primary Genre', fontsize=12)
plt.ylabel('Average Popularity', fontsize=12)

# Rotate x-axis labels
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Visualization - Histogram
import matplotlib.pyplot as plt
import seaborn as sns

# Create histogram for the most skewed numeric column
plt.figure(figsize=(10, 6))

sns.histplot(
    data=df_clean,
    x='popularity',
    bins=20,
    color='blue',
    edgecolor='black',
    kde=True
)

# Add title and labels
plt.title('Distribution of Movie Popularity', fontsize=16)
plt.xlabel('Popularity', fontsize=12)
plt.ylabel('Frequency', fontsize=12)

plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.show()

In [ ]:
# 7.Visualization - Scatter Plot

import matplotlib.pyplot as plt
import seaborn as sns

# Create scatter plot
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df_clean,
    x='budget',
    y='revenue',
    color='steelblue',
    alpha=0.6
)

# Add title and labels
plt.title('Relationship between Movie Budget and Revenue', fontsize=16)
plt.xlabel('Budget (USD)', fontsize=12)
plt.ylabel('Revenue (USD)', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.5)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create primary genre from the genres column if it doesn't exist
if 'primary_genre' not in df_clean.columns:
    df_clean['primary_genre'] = df_clean['genres'].str.split('|').str[0]

# Use df_clean as plot_df for consistency
plot_df = df_clean.copy()

# Fill NaN values in 'primary_genre' with 'Unknown' to avoid KeyError
plot_df['primary_genre'] = plot_df['primary_genre'].fillna('Unknown')

genre_counts = plot_df["primary_genre"].value_counts()

plot_df["genre_with_count"] = plot_df["primary_genre"].map(
    lambda genre: f"{genre} (n={genre_counts[genre]})"
)

# Define genre_order based on the counts to ensure consistent ordering
genre_order = genre_counts.index.tolist()

count_order = [
    f"{genre} (n={genre_counts[genre]})"
    for genre in genre_order
]

plt.figure(figsize=(16, 8)) # Increased figure width for x-axis labels

sns.boxplot(
    data=plot_df,
    x="genre_with_count", # Swapped x and y
    y="vote_average",     # Swapped x and y
    order=count_order,
    hue="genre_with_count",
    palette="Set2",
    legend=False,
    showfliers=False
)

plt.title(
    "Distribution of Movie Ratings by Primary Genre",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Primary Genre and Number of Movies") # Updated label
plt.ylabel("Average Movie Rating") # Updated label
plt.ylim(0, 10) # Set y-axis limit
plt.xticks(rotation=45, ha='right') # Rotate x-axis labels for readability
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# 8.Correlation Heat Map
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Select only numeric columns
numeric_df = df_clean.select_dtypes(include=['int64', 'float64'])

# Compute correlation matrix
corr_matrix = numeric_df.corr()

# Plot heatmap
plt.figure(figsize=(10, 8))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5
)

plt.title("Correlation Heatmap of Numeric Variables", fontsize=16)

plt.tight_layout()
plt.show()

# Find the highest absolute correlation (excluding self-correlation)
corr_abs = corr_matrix.abs()
np.fill_diagonal(corr_abs.values, 0)

highest_corr = corr_abs.unstack().sort_values(ascending=False).drop_duplicates()

print("Highest Absolute Correlation:")
print(highest_corr.head(1))

In [ ]:
# 9. Imputation Strategic Comparison

# Two most skewed numeric columns
skewed_columns = ['popularity', 'revenue']

for col in skewed_columns:
    mean_value = df_clean[col].mean()
    median_value = df_clean[col].median()

    print(f"\nColumn: {col}")
    print(f"Mean   : {mean_value:.2f}")
    print(f"Median : {median_value:.2f}")

    # Impute missing values using median
    df_clean[col] = df_clean[col].fillna(median_value)

# Verify that no missing values remain
print("\nMissing Values After Imputation:")
print(df_clean[skewed_columns].isnull().sum())

In [ ]:
# 9. Spearman Rank Correlation
import pandas as pd
import numpy as np

# Select only numeric columns
numeric_df = df_clean.select_dtypes(include=['int64', 'float64'])

# Pearson correlation matrix
pearson_corr = numeric_df.corr(method='pearson')

# Spearman correlation matrix
spearman_corr = numeric_df.corr(method='spearman')

# Print both matrices
print("Pearson Correlation Matrix")
print(pearson_corr)

print("\nSpearman Correlation Matrix")
print(spearman_corr)

# Difference between Spearman and Pearson
corr_difference = (spearman_corr - pearson_corr).abs()

# Remove duplicate pairs and self-correlations
mask = np.triu(np.ones(corr_difference.shape), k=1).astype(bool)

difference_table = (
    corr_difference.where(mask)
    .stack()
    .reset_index()
)

difference_table.columns = ['Variable 1', 'Variable 2', '|Spearman - Pearson|']

# Sort by largest difference
difference_table = difference_table.sort_values(
    by='|Spearman - Pearson|',
    ascending=False
)

print("\nDifference Table")
print(difference_table)

print("\nTop 3 Variable Pairs")
print(difference_table.head(3))

In [ ]:
# 9 Grouped Aggregation
import pandas as pd

# Create primary genre from the genres column
# df_clean['primary_genre'] = df_clean['genres'].str.split('|').str[0]

# Group by primary genre and compute statistics
grouped_stats = (
    df_clean.groupby('genres')['popularity']
    .agg(['mean', 'std', 'count'])
    .sort_values(by='mean', ascending=False)
)

print(grouped_stats)

# Find the group with the highest mean
highest_mean_group = grouped_stats['mean'].idxmax()
highest_mean = grouped_stats['mean'].max()

# Find the group with the highest standard deviation
highest_std_group = grouped_stats['std'].idxmax()
highest_std = grouped_stats['std'].max()

# Find the lowest group mean
lowest_mean_group = grouped_stats['mean'].idxmin()
lowest_mean = grouped_stats['mean'].min()

# Calculate the ratio
ratio = highest_mean / lowest_mean

print("\nGroup with Highest Mean:")
print(f"{highest_mean_group} ({highest_mean:.2f})")

print("\nGroup with Highest Standard Deviation:")
print(f"{highest_std_group} ({highest_std:.2f})")

print(f"\nHighest Mean / Lowest Mean Ratio: {ratio:.2f}")

In [ ]:
# 10. Save and verify Cleaned data

# Save the cleaned dataset to a CSV file
df_clean.to_csv('cleaned_data.csv', index=False)

print("Cleaned dataset saved successfully as 'cleaned_data.csv'.")

# Load the saved dataset to verify
df_verify = pd.read_csv('cleaned_data.csv')

# print(df_verify.head())
print(df_verify.shape)

In [ ]:
print(df_movie.dtypes)
print("/n")
print(df_clean.dtypes)